# Trabajo Práctico Integrador - Introducción al Análisis de Datos
## Entrega 6: Evaluación e Interpretación del Modelo de Clasificación

## Caso: análisis y predicción de cancelaciones en reservas hoteleras.

## Lenguaje utilizado: Python y SQL

## Propósito: El objetivo de esta entrega es evaluar e interpretar el modelo de árbol de decisión desarrollado para anticipar cancelaciones de reservas hoteleras.

# Datos de identificación

- **Apellido y nombre:** Rodriguez Joaquin - Pichulman Miguel
- **Comisión:** 12
- **Dataset asignado:** L
- **Entrega:** 6
- **Fecha:** 29 / 10 / 2026

# Preparación del entorno y reconstrucción del modelo

In [5]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

print("Preparando el entorno...")

for package in ["duckdb", "pandas", "scikit-learn"]:
    module = "sklearn" if package == "scikit-learn" else package
    if importlib.util.find_spec(module) is None:
        print(f"Instalando {package}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import duckdb
import pandas as pd

print("Paquetes preparados correctamente.")

DATA_URL = "https://raw.githubusercontent.com/joacoxr1/TPI-hotelbookinggrupoLOriginal/main/hotel%20booking%20TPI%20grupo%20L%20Original.csv"
DATA_PATH = Path("hotel_booking.csv")

if not DATA_PATH.exists():
    print("Descargando dataset...")
    urlretrieve(DATA_URL, DATA_PATH)

print("Cargando dataset...")
df_hotel = pd.read_csv(DATA_PATH)

# Si la celda se ejecuta otra vez, cerramos primero una conexión previa.
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

con = duckdb.connect(":memory:")
con.register("df_hotel_temp", df_hotel)
con.execute("CREATE OR REPLACE TABLE reservas AS SELECT * FROM df_hotel_temp")

reservas_count = f"{len(df_hotel):,}".replace(",", ".")
print(f"Entorno listo: {reservas_count} reservas y {df_hotel.shape[1]} variables.")

Preparando el entorno...
Paquetes preparados correctamente.
Cargando dataset...
Entorno listo: 25.000 reservas y 32 variables.


In [6]:
# Consulta SQL para seleccionar las variables
consulta = """
SELECT
    is_canceled,
    lead_time,
    market_segment,
    deposit_type,
    customer_type,
    previous_cancellations,
    hotel
FROM reservas
"""
df_modelo = con.sql(consulta).df()

# Definición de la variable objetivo (y) y las predictoras (X)
y = df_modelo["is_canceled"]
X = df_modelo.drop(columns=["is_canceled"])

# Separación entre entrenamiento y prueba
Evaluar el modelo con datos no utilizados durante la etapa de entrenamiento es fundamental para medir de forma fidedigna su capacidad de generalización sobre observaciones inéditas. Si evaluáramos el desempeño utilizando el mismo conjunto de datos empleado para el ajuste, correríamos el riesgo de medir únicamente memorización, sin garantizar que las reglas inferidas sean extrapolables a nuevas reservas.


In [7]:
# División del dataset en subconjuntos de train y test (70% / 30%)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Codificación de variables categóricas (Dummies)
categoricas = ["market_segment", "deposit_type", "customer_type", "hotel"]

import pandas as pd
X_train = pd.get_dummies(X_train, columns=categoricas, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categoricas, drop_first=True)

# Alineamos las columnas del conjunto de prueba con las de entrenamiento
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Matriz de confusión
A continuación, se entrena el árbol de decisión con profundidad acotada (max_depth=3) y se calculan las predicciones para contrastarlas con los valores reales del conjunto de evaluación.


In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
import pandas as pd

# Entrenamiento del modelo
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

# Predicción en el conjunto de prueba
y_pred = model.predict(X_test)

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Real: No Cancela (0)', 'Real: Cancela (1)'],
    columns=['Pred: No Cancela (0)', 'Pred: Cancela (1)']
)
print("Matriz de Confusión:")
print(cm_df)

Matriz de Confusión:
                      Pred: No Cancela (0)  Pred: Cancela (1)
Real: No Cancela (0)                  4739                  6
Real: Cancela (1)                     1850                905


### Interpretación de cuadrantes
Considerando que la clase positiva (1) representa la cancelación de la reserva:

Verdaderos Positivos (TP): Reservas que realmente fueron canceladas y que el modelo predijo correctamente como cancelaciones.
Verdaderos Negativos (TN): Reservas en las que los huéspedes efectivamente asistieron y el modelo anticipó correctamente que no se cancelarían.
Falsos Positivos (FP): Reservas en las que el cliente finalmente asistió al hotel, pero el modelo alertó erróneamente que se cancelaría.
Falsos Negativos (FN): Reservas que efectivamente terminaron siendo canceladas, pero el modelo predijo de forma equivocada que se concretarían sin problema.


# Métricas de clasificación

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Cálculo de métricas sobre el conjunto de test
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.7525
Precision: 0.9934
Recall:    0.3285
F1-score:  0.4937


### Análisis conceptual de métricas
- Accuracy (Exactitud): Representa la proporción total de clasificaciones correctas (tanto reservas concretadas como canceladas) sobre el universo total de reservas evaluadas.
- Precision (Precisión): Refleja la proporción de aciertos cuando el modelo emite una alerta de cancelación; indica cuán confiable es el clasificador cada vez que marca una reserva como positiva.
- Recall (Sensibilidad): Mide la proporción de cancelaciones reales que el modelo fue capaz de detectar de forma proactiva, visibilizando cuántos casos críticos se le escapan (falsos negativos).
- F1-score: Resume el balance armónico entre precision y recall, proveyendo una métrica integrada cuando existe asimetría de clases o costos dispares entre aciertos y falsas alarmas.


# Interpretación de los errores
La distinción cualitativa de los errores es clave para el negocio hotelero:

Falso Positivo (FP): El modelo predice que la reserva será cancelada, pero el huésped se presenta a realizar el check-in con normalidad.
Falso Negativo (FN): El modelo predice que la estadía se concretará sin cambios, pero el cliente termina cancelando.


## Impacto económico y operativo
Si bien el Falso Negativo produce una pérdida de ingresos directos (la habitación queda vacía a último momento sin posibilidad de reventa), el Falso Positivo puede resultar sustancialmente más costoso y crítico para la reputación de la empresa. Si la administración hotelera confía en la alerta de cancelación para sobreasignar o revender proactivamente la plaza (overbooking), la posterior llegada del cliente original obliga al establecimiento a incurrir en gastos extraordinarios de reubicación, compensaciones tarifarias y un daño severo a la fidelización del cliente y a la imagen de marca.


# Conclusión
El árbol de decisión resulta una herramienta sumamente valiosa para anticipar el riesgo de cancelación, respaldado por su capacidad de segmentar de forma interpretable factores determinantes como el tiempo de anticipación de la reserva (`lead_time`) o la exigencia de depósitos previos. No obstante, su implementación práctica debe considerar limitantes estructurales: por un lado, la asimetría de costos donde los falsos positivos conllevan riesgos reputacionales severos; por el otro, la naturaleza estática del modelo, que no contempla dinámicas exógenas como variaciones estacionales imprevistas, volatilidad macroeconómica, cancelaciones por factores climáticos o promociones repentinas de competidores.
